In [7]:
# indeed_click_apply.py
"""
Indeed Apply Button Clicker
---------------------------
✅ Loads saved login cookies if present
✅ If no cookies, opens login page and waits for manual login
✅ Detects successful login by presence of 'Account' button
✅ Saves cookies for future sessions
✅ Opens Indeed job page
✅ Clicks "Apply now" button automatically
"""

import os
import time
import pickle
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ==============================
# CONFIG
# ==============================
COOKIE_FILE = "indeed_cookies.pkl"
INDEED_HOME = "https://in.indeed.com/"
INDEED_LOGIN_URL = "https://secure.indeed.com/auth"
JOB_URL = "https://in.indeed.com/pagead/clk?mo=r&ad=-6NYlbfkN0DlNJszjx2cK18fvcYaDJXE9gEsQRH1e5F88H-kqtcRNdezuvX99gr_3D1dwPlYWw8lJSL7ATzAmGYts5LzPYuiDycGnBkDOTdrX_cJPHQw0frOL4XyvF0DtePui8XJRlwNj6s7Bd2vOuMK-PxrnQs85UEVdJQCoW1sKCIBGrnupzsVdCyD2s7cSE2ZbmGRrh3iPbp59A5c4ZT59lhDXhgob20ydyzkt58cVDMy72BhAq6FXHQl-eks9lgdjO28jkxtHkw-91CnZgKQeh5OQ8Jh84I78Y_TOzL7kFRwDlexbk6QNRSQh8k46-b_bxGMN0z9naEhOJKcNd-uvh98kLcIZSoJ6H8BXvFKNU7Gid2-BmWvszMpf7g-7vMRr4ObTC5hBS7XApowBNEUDGHQ-07-4FrEsX9pbBY3S06fyNxXBZb-tO4ZKcH53YY5vw3cA9VFjf6eU1CGha_dXG-coPIacfgZCAAILgX3GUtUCUQy66Ne90rceaJpO3BROQPlZiVjUChoXzWEU4TvBapUOExKyvYtoZ_j3YeBuh3oUw5gct4PzWBXfmdJsaziQ8iKXfKZ76I4zMhfXPkMXqrf_eOUF5bcokndWIccBaQsDK7Fa7mAvVGhUOfjfGwSzSM1uuiDSXjSp25979vAvheaYQGmQYG6aDUfdIqruqhcfSjVyfq-Jc-TlmUl&xkcb=SoB-6_M3rVJ_mlzNXJ0LbzkdCdPP&camk=UoKtGZLa3XKNaD21pfAMmA==&p=0&fvj=0&vjs=3"

# ==============================
# DRIVER SETUP
# ==============================
def setup_driver(headless=False):
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")

    options.add_argument("--no-sandbox")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    })
    return driver

# ==============================
# COOKIE HELPERS
# ==============================
def save_cookies(driver):
    """Save cookies to file."""
    with open(COOKIE_FILE, "wb") as f:
        pickle.dump(driver.get_cookies(), f)
    print("💾 Cookies saved successfully!")


def load_cookies(driver):
    """Load cookies from file if available."""
    if not os.path.exists(COOKIE_FILE):
        return False

    driver.get(INDEED_HOME)
    with open(COOKIE_FILE, "rb") as f:
        cookies = pickle.load(f)

    for cookie in cookies:
        try:
            driver.add_cookie(cookie)
        except Exception:
            pass

    driver.refresh()
    print("🍪 Cookies loaded successfully.")
    time.sleep(2)
    return True

# ==============================
# LOGIN HANDLER
# ==============================
def handle_login(driver):
    """
    If cookies not found, prompt user to log in manually.
    Detect successful login via Account button.
    """
    print("🔐 No cookies found. Opening login page for manual login...")
    driver.get(INDEED_LOGIN_URL)

    try:
        # Wait until Account button is visible, means login successful
        print("⏳ Please log in manually. Waiting for login to complete...")
        WebDriverWait(driver, 300).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "button#AccountMenu"))
        )
        print("✅ Login successful!")
        save_cookies(driver)
        return True
    except Exception:
        print("❌ Login timeout or failed. Please try again.")
        return False

# ==============================
# MAIN APPLY LOGIC
# ==============================
def click_apply_button(driver, job_url):
    driver.get(job_url)
    print(f"🔗 Opened job: {job_url}")

    try:
        # Wait for Apply button
        apply_btn = WebDriverWait(driver, 20).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button#indeedApplyButton"))
        )
        driver.execute_script("arguments[0].scrollIntoView(true);", apply_btn)
        time.sleep(1)
        driver.execute_script("arguments[0].click();", apply_btn)
        print("✅ Clicked 'Apply now' button successfully!")
        time.sleep(3)
    except Exception as e:
        print(f"❌ Could not find or click apply button: {e}")

# ==============================
# MAIN ENTRY
# ==============================
if __name__ == "__main__":
    driver = setup_driver(headless=False)

    if not load_cookies(driver):
        if not handle_login(driver):
            print("⚠️ Could not complete login. Exiting.")
            driver.quit()
            exit()

    click_apply_button(driver, JOB_URL)

    print("🏁 Script finished.")
    driver.quit()


🍪 Cookies loaded successfully.
🔗 Opened job: https://in.indeed.com/pagead/clk?mo=r&ad=-6NYlbfkN0DlNJszjx2cK18fvcYaDJXE9gEsQRH1e5F88H-kqtcRNdezuvX99gr_3D1dwPlYWw8lJSL7ATzAmGYts5LzPYuiDycGnBkDOTdrX_cJPHQw0frOL4XyvF0DtePui8XJRlwNj6s7Bd2vOuMK-PxrnQs85UEVdJQCoW1sKCIBGrnupzsVdCyD2s7cSE2ZbmGRrh3iPbp59A5c4ZT59lhDXhgob20ydyzkt58cVDMy72BhAq6FXHQl-eks9lgdjO28jkxtHkw-91CnZgKQeh5OQ8Jh84I78Y_TOzL7kFRwDlexbk6QNRSQh8k46-b_bxGMN0z9naEhOJKcNd-uvh98kLcIZSoJ6H8BXvFKNU7Gid2-BmWvszMpf7g-7vMRr4ObTC5hBS7XApowBNEUDGHQ-07-4FrEsX9pbBY3S06fyNxXBZb-tO4ZKcH53YY5vw3cA9VFjf6eU1CGha_dXG-coPIacfgZCAAILgX3GUtUCUQy66Ne90rceaJpO3BROQPlZiVjUChoXzWEU4TvBapUOExKyvYtoZ_j3YeBuh3oUw5gct4PzWBXfmdJsaziQ8iKXfKZ76I4zMhfXPkMXqrf_eOUF5bcokndWIccBaQsDK7Fa7mAvVGhUOfjfGwSzSM1uuiDSXjSp25979vAvheaYQGmQYG6aDUfdIqruqhcfSjVyfq-Jc-TlmUl&xkcb=SoB-6_M3rVJ_mlzNXJ0LbzkdCdPP&camk=UoKtGZLa3XKNaD21pfAMmA==&p=0&fvj=0&vjs=3
✅ Clicked 'Apply now' button successfully!
🏁 Script finished.
